In [1]:
# Parameters
run_date = "2026-01-01"  # papermill replacement
import os
output_dir = os.environ.get("ORION_SIGNALS_DIR", "../signals")
dry_run = False

# ensure output exists
os.makedirs(output_dir, exist_ok=True)


In [2]:
# Import basic modules
import pandas as pd
import numpy as np
import math
import datetime
from datetime import timedelta
from typing import Optional, List, Dict, Any

# Import warnings
import warnings
warnings.filterwarnings("ignore")


In [3]:
from __future__ import annotations

# -- PullBack: fade a stack that stands too far from yesterday's close in the early pre-market ----------
#
# THE TRADE (operator's own spec, 2026-10-18): "тікери які різко ростуть чи падають після 4:04. Більше
# ніж на 0.7%, події які почалися до точки початку і на момент 4:04 вже стоять зависоко чи занизько теж
# розглядаємо ... тікер різко відріст чи впав і такий рух можна розвернути і вийти гепом. Розвертати
# або в 4:30, 5:00 або в 5:30 ... час початку розвороту це 3 класи ... входом у цей час +-5хв якщо
# немає конкретної хвилини і виходом в 9:30."
#
# Concretely: 04:04 is the REFERENCE minute (not 04:00 - the 03:58-04:02 window is a known dropped-print
# zone in this data set and must never become a price). From it, at each of three reversal checkpoints
# (04:30 / 05:00 / 05:30, +/-5 min), read the stack's Stack%: on this axis that IS its displacement from
# yesterday's regular-session close. Fade that displacement - Short what stands too high, Long what
# stands too low - and hold to the 09:30 open. One exit for all three classes; what the classes vary is
# WHEN THE REVERSAL IS STARTED, which is the question the operator asked to analyse.
#
# TWO ADMISSION TRIGGERS, because the operator asked for both kinds of event and they are not the same:
#
#   LEVEL  |Stack%(entry)| >= min_dev
#          The stack already stands too far out at the entry minute, however it got there - this is the
#          "події які почалися до точки початку" half, the move that began before 04:04.
#   MOVE   |Stack%(entry) - Stack%(04:04)| >= min_dev  AND it points the SAME WAY as the displacement
#          A fresh run after 04:04 that pushed the stack further out - the "різко ростуть чи падають
#          після 4:04" half. The same-direction requirement is load-bearing: a stack sitting at -1.0%
#          that rallies +1.5% to +0.5% has moved hard, but fading it would be SHORTING something that
#          just rose off a low, which is a ride, not a pullback. Without the check those rows enter with
#          the wrong thesis and quietly dilute every rating.
#
# A row records which trigger admitted it (`kind` = level / move / both), so the analysis can ask
# whether the two populations behave the same - they are different claims and deserve separate numbers.
#
# THE TRADED SIGNAL IS ALWAYS THE DISPLACEMENT, never the move. The position is "this stack is too far
# from its close and the open will pull it back", so the thing being faded, and the thing the 09:30
# print resolves against, is Stack%(entry). The move only decides admission.
#
# ALPHA / GAMMA / LAMBDA ARE THIS STRATEGY'S OWN, NOT THE FAMILY'S (operator, 2026-10-18 - this
# deliberately overrides the "lambda is the same statistic in every family" rule of 2026-10-11 for
# PullBack only):
#
#   ALPHA   per ticker PER CLASS, split pos/neg: the RMS of how the stack stood at that class's own
#           checkpoint, over every session it printed there. Gate-free.
#   GAMMA   per ticker PER CLASS, split pos/neg: the MAXIMUM the stack reached between that class's
#           checkpoint and 09:30 - how far the position can be expected to go against (gamma_pos for a
#           short) or in favour of it.
#   LAMBDA  per ticker, ONE value for all classes: the RMS of the per-session maximum |Stack%| over
#           04:04..09:30, counting only sessions where that maximum reached min_dev.
#
# The raw material for all three is collected here in scan_day and reduced in the ratings cell, so this
# file stays the only place that knows the clock.
import math
from collections import namedtuple

import numpy as np
import pandas as pd

SESSION_SPLIT_MIN = 900   # 15:00 - the family's own stage split. Every PullBack checkpoint (04:04..09:30)
                          # sits far before it, so nothing here is ever treated as "late"/next-session.
                          # Kept because the stage still has to map a timestamp onto a session axis.

Checkpoint = namedtuple("Checkpoint", "name smin tol")

# The REFERENCE minute. Its tolerance is searched FORWARD-first and the dropped window is excluded
# outright, so in practice the reference is the first clean print at or after 04:03.
BASE_SMIN = 4 * 60 + 4    # 04:04
BASE_TOL = 5

# 03:58-04:02 inclusive: a known dropped-print window in this data set. Never accepted as a price, even
# if a finite number sits there.
DROPPED_LO_SMIN = 3 * 60 + 58
DROPPED_HI_SMIN = 4 * 60 + 2
DROPPED_SMINS = frozenset(range(DROPPED_LO_SMIN, DROPPED_HI_SMIN + 1))

SEARCH_WINDOW_MIN = 5     # the operator's own "+-5хв якщо немає конкретної хвилини"
_RAW_CHECKPOINTS = (
    ("T0430", (4, 30), SEARCH_WINDOW_MIN),
    ("T0500", (5, 0), SEARCH_WINDOW_MIN),
    ("T0530", (5, 30), SEARCH_WINDOW_MIN),
    ("EXIT", (9, 30), SEARCH_WINDOW_MIN),
)
ENTRY_CLASSES = ("T0430", "T0500", "T0530")

# Entry and exit land the SAME calendar morning - no midnight crossing anywhere in this strategy.
EXIT_DATE_OFFSET = {"T0430": 0, "T0500": 0, "T0530": 0}


def _to_smin(hm, split=SESSION_SPLIT_MIN):
    t = hm[0] * 60 + hm[1]
    return t - 1440 if t >= split else t


def build_checkpoints(split=SESSION_SPLIT_MIN):
    """The three entry classes + the shared 09:30 exit, each tolerance capped at half the gap to its
    nearest neighbour so two checkpoints can never claim the same print. The real gaps here (30 min
    between entries, 4 hours to the exit) never actually bind the operator's +/-5."""
    smins = [_to_smin(hm, split) for _, hm, _ in _RAW_CHECKPOINTS]
    out = {}
    for i, (name, hm, req) in enumerate(_RAW_CHECKPOINTS):
        prev_gap = (smins[i] - smins[i - 1]) if i > 0 else math.inf
        next_gap = (smins[i + 1] - smins[i]) if i + 1 < len(smins) else math.inf
        tol = min(req, prev_gap // 2, next_gap // 2)
        out[name] = Checkpoint(name, smins[i], int(tol))
    return out


def default_cfg(**over):
    cfg = dict(
        min_dev=2.0,          # RAISED from 0.7 to 2.0 on the operator's instruction, 2026-10-18
                              # ("тоді поріг 0.7 заміни на 2 всюди"), after the first real run showed the
                              # 0.7 floor admitting ~190 trades/day/class whose own |displacement|
                              # bands were flat-to-negative: at T0500, 0.7-1 gave -0.04 and 1-1.5
                              # gave -0.03, while 4-7 gave +0.47 and 7+ gave +1.31.
                              # ONE KNOB, THREE USES - raising it moves all three together:
                              #   1. the LEVEL admission trigger,
                              #   2. the MOVE admission trigger,
                              #   3. the floor LAMBDA filters sessions by, so lambda now describes
                              #      only sessions where the stack reached 2pp, not 0.7pp.
        alpha_min_n=3,        # sessions needed before an ALPHA is published for a (ticker, class, sign)
        alpha_trim_q=0.99,    # the operator's "ролінг аномалій": the largest |readings| are dropped
                              # before the RMS, so one broken print cannot define a ticker's alpha
        gamma_min_n=3,        # sessions needed before a GAMMA is published
        lambda_min_n=2,       # qualifying sessions needed before a LAMBDA is published
        max_abs_pp=60.0,      # a print beyond this is a broken quote, not a price
    )
    cfg.update(over)
    return cfg


def panel_from_stack(tickers, t_first, stack):
    """One value per ticker per minute - Stack% itself. `t_first` is the SMIN of column 0."""
    return dict(tickers=np.asarray(tickers), t_first=int(t_first), stack=np.asarray(stack, np.float32))


def _clean(stack, cfg):
    bad = ~np.isfinite(stack) | (np.abs(stack) > cfg["max_abs_pp"])
    out = stack.copy()
    out[bad] = np.nan
    return out


def _nearest(mat, t_first, t_col, tol, exclude=None, prefer_later=False):
    """value + the absolute minute actually used, per row, searching outward from t_col: 0, -1, +1, -2,
    +2, ... (or 0, +1, -1, +2, -2, ... when `prefer_later=True`). The first ring that finds a print wins
    and a row already filled is never looked at again.

    `exclude`: raw minute indices never accepted even if finite - PullBack uses it for the 03:58-04:02
    dropped window, which is the only reason this parameter is not dead weight here.

    Sentinel: a row with NO match anywhere in the window gets `minute == -1` (the literal integer).
    """
    N, T = mat.shape
    val = np.full(N, np.nan, np.float32)
    used = np.full(N, -1, np.int64)
    remaining = np.arange(N)
    j0 = t_col - t_first
    excl_j = {m - t_first for m in exclude} if exclude else set()
    for d in range(0, int(tol) + 1):
        if remaining.size == 0:
            break
        order = (j0,) if d == 0 else ((j0 + d, j0 - d) if prefer_later else (j0 - d, j0 + d))
        for j in order:
            if remaining.size == 0:
                break
            if j < 0 or j >= T or j in excl_j:
                continue
            v = mat[remaining, j]
            hit = np.isfinite(v)
            if hit.any():
                idx = remaining[hit]
                val[idx] = v[hit]
                used[idx] = j
                remaining = remaining[~hit]
    minute = np.where(used >= 0, used + t_first, -1)
    return val, minute


def _window_extremes(mat, t_first, lo_min, hi_min):
    """(highest, lowest) finite Stack% per ticker over the INCLUSIVE minute range [lo_min, hi_min].

    GAMMA's and LAMBDA's raw material. Unlike `_nearest` this reads every minute in the window, not one
    checkpoint - "максимальне значення як стак стояв на проміжку від початку класу ... до 9:30" is a
    property of the whole stretch, not of any single print. A ticker with no print at all in the window
    gets NaN on both sides, never 0.
    """
    lo_j = max(0, lo_min - t_first)
    hi_j = min(mat.shape[1] - 1, hi_min - t_first)
    if lo_j > hi_j:
        n = mat.shape[0]
        return np.full(n, np.nan, np.float32), np.full(n, np.nan, np.float32)
    sl = mat[:, lo_j:hi_j + 1]
    with np.errstate(all="ignore"):
        any_finite = np.isfinite(sl).any(axis=1)
        hi = np.where(any_finite, np.nanmax(np.where(np.isfinite(sl), sl, -np.inf), axis=1), np.nan)
        lo = np.where(any_finite, np.nanmin(np.where(np.isfinite(sl), sl, np.inf), axis=1), np.nan)
    return hi.astype(np.float32), lo.astype(np.float32)


_EMPTY_TRADES = pd.DataFrame(columns=["ticker", "cls", "sign", "kind", "entry_dev", "base_dev", "move_dev",
                                      "entry_val", "exit_val", "pnl", "base_minute", "entry_minute",
                                      "exit_minute", "date_offset"])


def scan_day(panel, cfg, checkpoints):
    """-> (trades, signals, excursions, session_max) for one session (one calendar morning).

    trades     : one row per ticker per ENTRY CLASS that had a 04:04 reference, a valid entry for that
                 class, a valid 09:30 exit, and cleared either admission trigger.
    signals    : the RAW displacement at every class checkpoint for every ticker that printed one, gated
                 on nothing - ALPHA's raw material, carrying `cls` because ALPHA is per class here.
    excursions : per (ticker, class) the highest and lowest Stack% between that class's checkpoint and
                 09:30 - GAMMA's raw material. Gate-free: a stack that never traded still has an
                 excursion, and gamma is a statement about the stack, not about the trades.
    session_max: per ticker the largest |Stack%| anywhere in 04:04..09:30 - LAMBDA's raw material.
    """
    stack = _clean(panel["stack"], cfg)
    t_first = panel["t_first"]
    tk = panel["tickers"]
    exit_smin = checkpoints["EXIT"].smin

    # The 04:04 reference. prefer_later + the dropped-window exclusion make this "the first clean print
    # at or after 04:03" in every realistic case.
    base_v, base_m = _nearest(stack, t_first, BASE_SMIN, BASE_TOL,
                              exclude=DROPPED_SMINS, prefer_later=True)
    ex_v, ex_m = _nearest(stack, t_first, exit_smin, checkpoints["EXIT"].tol, prefer_later=True)

    # LAMBDA's raw material: the session's own largest excursion either way, 04:04 -> 09:30. The
    # min_dev floor is applied in the ratings cell, not here, so this table stays the full gate-free
    # population and a later change of threshold needs no restage.
    s_hi, s_lo = _window_extremes(stack, t_first, BASE_SMIN, exit_smin)
    with np.errstate(all="ignore"):
        s_abs = np.fmax(np.abs(s_hi), np.abs(s_lo))
    s_ok = np.isfinite(s_abs)
    session_max = pd.DataFrame({"ticker": tk[s_ok], "max_abs": s_abs[s_ok],
                                "max_hi": s_hi[s_ok], "max_lo": s_lo[s_ok]})

    sig_rows, exc_rows, rows = [], [], []
    for cls in ENTRY_CLASSES:
        ck = checkpoints[cls]
        ent_v, ent_m = _nearest(stack, t_first, ck.smin, ck.tol, prefer_later=False)
        ent_ok = np.isfinite(ent_v)
        if ent_ok.any():
            sig_rows.append(pd.DataFrame({"ticker": tk[ent_ok], "cls": cls, "dev": ent_v[ent_ok]}))

        # GAMMA's raw material for this class: how far the stack ran, either way, from this class's own
        # checkpoint to the 09:30 exit.
        g_hi, g_lo = _window_extremes(stack, t_first, ck.smin, exit_smin)
        g_ok = np.isfinite(g_hi) | np.isfinite(g_lo)
        if g_ok.any():
            exc_rows.append(pd.DataFrame({"ticker": tk[g_ok], "cls": cls,
                                          "exc_hi": g_hi[g_ok], "exc_lo": g_lo[g_ok]}))

        dev = ent_v
        move = ent_v - base_v
        level_ok = np.abs(dev) >= cfg["min_dev"]
        # Same-direction requirement - see this file's own header comment for why a big move AGAINST the
        # displacement must not admit the row.
        move_ok = (np.abs(move) >= cfg["min_dev"]) & (np.sign(move) == np.sign(dev)) & (dev != 0)
        have = ent_ok & np.isfinite(base_v) & np.isfinite(ex_v) & (level_ok | move_ok)
        if not have.any():
            continue
        j = np.flatnonzero(have)

        e_dev = dev[j]
        e_val = ent_v[j]
        x_val = ex_v[j]
        # Fade the displacement: Short what stands too high (dev > 0), Long what stands too low. The
        # 09:30 open is the exit for every class.
        pnl = -np.sign(e_dev) * (x_val - e_val)
        kind = np.where(level_ok[j] & move_ok[j], "both", np.where(level_ok[j], "level", "move"))
        rows.append(pd.DataFrame({
            "ticker": tk[j], "cls": cls, "sign": np.where(e_dev > 0, "pos", "neg"), "kind": kind,
            "entry_dev": e_dev, "base_dev": base_v[j], "move_dev": move[j],
            "entry_val": e_val, "exit_val": x_val, "pnl": pnl.astype(np.float32),
            "base_minute": base_m[j], "entry_minute": ent_m[j], "exit_minute": ex_m[j],
            "date_offset": EXIT_DATE_OFFSET[cls],
        }))

    trades = pd.concat(rows, ignore_index=True) if rows else _EMPTY_TRADES.copy()
    signals = pd.concat(sig_rows, ignore_index=True) if sig_rows else pd.DataFrame(
        columns=["ticker", "cls", "dev"])
    excursions = pd.concat(exc_rows, ignore_index=True) if exc_rows else pd.DataFrame(
        columns=["ticker", "cls", "exc_hi", "exc_lo"])
    return trades, signals, excursions, session_max


def _add_days(sdate_int, offset_days):
    """int YYYYMMDD + a signed day offset -> int YYYYMMDD. Always a no-op here (EXIT_DATE_OFFSET is all
    zeros) - kept for shape parity with the sibling families."""
    if offset_days == 0:
        return sdate_int
    ts = pd.Timestamp(str(sdate_int)) + pd.Timedelta(days=int(offset_days))
    return int(ts.strftime("%Y%m%d"))


def scan_days(day_iter, cfg, checkpoints=None, progress_every=10):
    checkpoints = checkpoints or build_checkpoints()
    tr_all, sg_all, ex_all, sm_all = [], [], [], []
    for n, (sdate, panel) in enumerate(day_iter, 1):
        tr, sg, ex, sm = scan_day(panel, cfg, checkpoints)
        for df, bucket in ((tr, tr_all), (sg, sg_all), (ex, ex_all), (sm, sm_all)):
            if len(df):
                df = df.copy()
                df.insert(0, "sdate", int(sdate))
                bucket.append(df)
        if len(tr):
            tr_all[-1]["trade_date"] = [_add_days(int(sdate), off) for off in tr_all[-1]["date_offset"]]
            tr_all[-1] = tr_all[-1].drop(columns="date_offset")
        if progress_every and n % progress_every == 0:
            print(f"  scanned {n} sessions (last {sdate}) trades so far {sum(len(x) for x in tr_all):,}",
                  flush=True)
    trades = pd.concat(tr_all, ignore_index=True) if tr_all else (
        _EMPTY_TRADES.copy().assign(sdate=pd.Series(dtype=int), trade_date=pd.Series(dtype=int))
        [["sdate", *[c for c in _EMPTY_TRADES.columns if c != "date_offset"], "trade_date"]])
    signals = pd.concat(sg_all, ignore_index=True) if sg_all else pd.DataFrame(
        columns=["sdate", "ticker", "cls", "dev"])
    excursions = pd.concat(ex_all, ignore_index=True) if ex_all else pd.DataFrame(
        columns=["sdate", "ticker", "cls", "exc_hi", "exc_lo"])
    session_max = pd.concat(sm_all, ignore_index=True) if sm_all else pd.DataFrame(
        columns=["sdate", "ticker", "max_abs", "max_hi", "max_lo"])
    return trades, signals, excursions, session_max


In [4]:
from __future__ import annotations

# -- ratings, alpha, gamma, lambda ---------------------------------------------------------------------
#
# ALL THREE LEVELS ARE THIS STRATEGY'S OWN (operator, 2026-10-18). They do NOT match the OPG/CLO
# families' definitions, and that is deliberate - including LAMBDA, which the operator's own earlier
# rule (2026-10-11) said was identical everywhere. For PullBack it is not. Spelled out, because a reader
# coming from a sibling notebook will otherwise assume the familiar meaning:
#
#   ALPHA   "середнє квадратичне значення того як стояв стак на порозі кожного класу ... для позитивних
#           і негативних альфа окрема"
#           -> per (ticker, CLASS, sign): RMS of the displacement at that class's own checkpoint, over
#              every session the ticker printed there. Gate-free: every reading counts, not just the
#              ones big enough to trade. PER CLASS, so T0430 / T0500 / T0530 each get their own pair.
#           Sibling families instead publish ONE modal value per ticker - different statistic, different
#           granularity.
#
#   GAMMA   "максимальне значення як стак стояв на проміжку від початку класу наприклад 5:00 до 9:30,
#           шукаємо окремо позитивні і негативні"
#           -> per (ticker, CLASS, sign): the MAXIMUM the stack reached between that class's checkpoint
#              and 09:30, positives and negatives separately. Sibling families instead walk a 95%
#              lower-bound search for the lowest PROFITABLE entry level - a completely different number
#              that happens to share the name.
#           A raw maximum is defined by ONE session, so `gamma_p90` and `gamma_n` ship beside it; read
#              those before sizing anything off gamma.
#
#   LAMBDA  "Для усіх класів однакова максимальні значення за сесію від 4:04 до 9:30, але розглядаємо
#           лише ті ситуації де стак стояв від [min_dev]% і розраховуємо середнє квадратичне"
#           -> per ticker, ONE value for all classes: RMS of the per-session maximum |Stack%| over
#              04:04..09:30, counting only sessions whose maximum reached min_dev. That floor was 0.7
#              when the statistic was specified and is 2.0 since the operator raised the trigger
#              "всюди" (2026-10-18) - lambda therefore now describes a strictly rarer set of
#              sessions than the first published run did, which is why lambda_n ships beside it.
#
#   SIGMA   read from final.parquet's own column, unchanged - a ticker property, not strategy-specific.
#
# "ролінг аномалій" (the operator's phrase) is implemented as a TRIM: before any RMS, the largest
# |readings| above `alpha_trim_q` are dropped, so a single broken print cannot define a ticker's level.
import math

import numpy as np
import pandas as pd


def _wilson_lb(k, n, z=1.96):
    k = np.asarray(k, float)
    n = np.asarray(n, float)
    with np.errstate(all="ignore"):
        p = k / n
        den = 1 + z * z / n
        c = p + z * z / (2 * n)
        r = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
        return np.where(n > 0, (c - r) / den, np.nan)


def _rms(values, trim_q=0.99, min_n=1):
    """Root-mean-square of |values| after the anomaly trim. NaN below min_n readings.

    The trim drops readings whose magnitude exceeds the trim_q quantile - the operator's "ролінг
    аномалій". It only engages once there are enough readings for a quantile to mean anything (>=10),
    so a ticker with four sessions keeps all four.
    """
    v = np.asarray(values, float)
    v = np.abs(v[np.isfinite(v)])
    if v.size < min_n or v.size == 0:
        return float("nan")
    if v.size >= 10 and 0 < trim_q < 1:
        cap = np.quantile(v, trim_q)
        kept = v[v <= cap]
        if kept.size:
            v = kept
    return float(np.sqrt(np.mean(v ** 2)))


def compute_alpha(signals, cfg):
    """ALPHA per (ticker, cls, sign) - see this cell's own header.

    -> DataFrame [ticker, cls, alpha_pos, alpha_n_pos, alpha_neg, alpha_n_neg]. A side with fewer than
    cfg["alpha_min_n"] sessions is NaN rather than a number built from two readings.
    """
    cols = ["ticker", "cls", "alpha_pos", "alpha_n_pos", "alpha_neg", "alpha_n_neg"]
    if not len(signals):
        return pd.DataFrame(columns=cols)
    out = []
    q, min_n = cfg["alpha_trim_q"], cfg["alpha_min_n"]
    for (tk, cls), g in signals.groupby(["ticker", "cls"], sort=False):
        dev = g["dev"].to_numpy(float)
        pos, neg = dev[dev > 0], dev[dev < 0]
        out.append((tk, cls,
                    _rms(pos, q, min_n), int(pos.size),
                    _rms(neg, q, min_n), int(neg.size)))
    return pd.DataFrame(out, columns=cols)


def compute_gamma(excursions, cfg):
    """GAMMA per (ticker, cls, sign) - see this cell's own header.

    gamma_pos is the highest the stack ever reached between the class checkpoint and 09:30; gamma_neg is
    the magnitude of the lowest it ever reached. Both are maxima over sessions of a per-session extreme,
    so each is defined by a SINGLE session - `gamma_p90` (the 90th percentile of the per-session
    extremes) and `gamma_n` are published beside them for exactly that reason.

    -> DataFrame [ticker, cls, gamma_pos, gamma_pos_p90, gamma_neg, gamma_neg_p90, gamma_n].
    """
    cols = ["ticker", "cls", "gamma_pos", "gamma_pos_p90", "gamma_neg", "gamma_neg_p90", "gamma_n"]
    if not len(excursions):
        return pd.DataFrame(columns=cols)
    min_n = cfg["gamma_min_n"]
    out = []
    for (tk, cls), g in excursions.groupby(["ticker", "cls"], sort=False):
        hi = g["exc_hi"].to_numpy(float)
        lo = g["exc_lo"].to_numpy(float)
        n = int(np.isfinite(hi).sum())
        if n < min_n:
            continue
        up = hi[np.isfinite(hi) & (hi > 0)]
        dn = -lo[np.isfinite(lo) & (lo < 0)]
        out.append((
            tk, cls,
            float(up.max()) if up.size else float("nan"),
            float(np.quantile(up, 0.90)) if up.size else float("nan"),
            float(dn.max()) if dn.size else float("nan"),
            float(np.quantile(dn, 0.90)) if dn.size else float("nan"),
            n,
        ))
    return pd.DataFrame(out, columns=cols)


def compute_lambda(session_max, cfg):
    """LAMBDA per ticker - see this cell's own header. One value for every class.

    Only sessions whose own maximum |Stack%| over 04:04..09:30 reached cfg["min_dev"] are counted, which
    is the operator's "розглядаємо лише ті ситуації де стак стояв від [min_dev]%" (0.7 when the statistic
    was specified, 2.0 since): lambda describes how big
    this stack's moves are WHEN it moves, not how quiet it is on an average day.

    -> DataFrame [ticker, lambda, lambda_n].
    """
    cols = ["ticker", "lambda", "lambda_n"]
    if not len(session_max):
        return pd.DataFrame(columns=cols)
    q = session_max[session_max["max_abs"] >= cfg["min_dev"]]
    if not len(q):
        return pd.DataFrame(columns=cols)
    out = []
    for tk, g in q.groupby("ticker", sort=False):
        v = g["max_abs"].to_numpy(float)
        if v.size < cfg["lambda_min_n"]:
            continue
        out.append((tk, _rms(v, cfg["alpha_trim_q"], cfg["lambda_min_n"]), int(v.size)))
    return pd.DataFrame(out, columns=cols)


def compute_ratings(trades):
    """total/wins/win_rate(+Wilson LB)/mean pnl(+95% LB)/median, per (ticker, cls, sign) - the three
    reversal-start classes are exactly what `cls` carries, which is the operator's own question."""
    if not len(trades):
        return pd.DataFrame(columns=["ticker", "cls", "sign", "total", "wins", "win_rate", "win_rate_lb",
                                     "mean_pnl", "median_pnl", "pnl_lb"])
    g = trades.groupby(["ticker", "cls", "sign"])
    out = g["pnl"].agg(total="size", mean_pnl="mean", median_pnl="median", sd_pnl="std").reset_index()
    out["wins"] = g["pnl"].apply(lambda s: int((s > 0).sum())).to_numpy()
    out["win_rate"] = out["wins"] / out["total"]
    out["win_rate_lb"] = _wilson_lb(out["wins"], out["total"])
    with np.errstate(all="ignore"):
        se = out["sd_pnl"] / np.sqrt(out["total"])
        out["pnl_lb"] = np.where(out["total"] > 1, out["mean_pnl"] - 1.96 * se, np.nan)
    return out.drop(columns="sd_pnl")


def build_summary(trades, signals, excursions, session_max, cfg, sigma_by_ticker=None):
    """ratings + the three levels, joined into one per-(ticker, cls, sign) table.

    "pos"/"neg" label the DIRECTION OF THE DISPLACEMENT being faded, not the position taken - so a "pos"
    row is a SHORT (the stack stood too high). alpha and gamma are SIGN-MATCHED to that label: a pos row
    carries alpha_pos/gamma_pos. lambda and sigma are ticker-level scalars and appear unchanged on every
    row of that ticker.
    """
    alpha = compute_alpha(signals, cfg)
    gamma = compute_gamma(excursions, cfg)
    lam = compute_lambda(session_max, cfg)
    ratings = compute_ratings(trades)
    if not len(ratings):
        empty = ratings.copy()
        for c in ("alpha", "alpha_n", "gamma", "gamma_p90", "lambda", "lambda_n", "sigma"):
            empty[c] = np.nan
        return empty, alpha, gamma, lam

    out = ratings.merge(alpha, on=["ticker", "cls"], how="left").merge(gamma, on=["ticker", "cls"], how="left")
    pos = out["sign"].to_numpy() == "pos"
    out["alpha"] = np.where(pos, out["alpha_pos"], out["alpha_neg"])
    out["alpha_n"] = np.where(pos, out["alpha_n_pos"], out["alpha_n_neg"])
    out["gamma"] = np.where(pos, out["gamma_pos"], out["gamma_neg"])
    out["gamma_p90"] = np.where(pos, out["gamma_pos_p90"], out["gamma_neg_p90"])
    out = out.drop(columns=["alpha_pos", "alpha_neg", "alpha_n_pos", "alpha_n_neg",
                            "gamma_pos", "gamma_neg", "gamma_pos_p90", "gamma_neg_p90"])

    out = out.merge(lam, on="ticker", how="left")
    out["sigma"] = out["ticker"].map(sigma_by_ticker) if sigma_by_ticker else np.nan
    return out, alpha, gamma, lam


In [5]:
from __future__ import annotations

# -- stage: final.parquet -> one small parquet per SESSION ---------------------------------------------------
# Same staging strategy the OPG/CLO families use - PullBack reads the exact same final.parquet and the exact
# same Stack%/sigma columns. Renamed (pullback_) so every strategy's own stage directory and sidecar live on
# disk without colliding.
#
# THE KEPT BAND is 03:55..09:40 and nothing else. Unlike the sibling families there is NO prior-afternoon
# slice: PullBack's lambda is a morning statistic (the per-session maximum over 04:04..09:30), not the
# family's last-print-before-the-close one, so the 15:5x minutes those notebooks stage would be dead
# weight here. The band is wider than the five checkpoints this strategy reads because GAMMA and LAMBDA
# need EVERY minute between a class and 09:30, not just the checkpoints.
import numpy as np
import pandas as pd

# SESSION_SPLIT_MIN and panel_from_stack come from this notebook's own core cell, executed into the same
# namespace before this one (notebook order) - not imported, the same pattern every sibling notebook uses.

STAGE_SMIN_LO = 235   # 03:55
STAGE_SMIN_HI = 580   # 09:40


def pullback_stage_by_session(input_path, stage_dir, *, session_split_min=SESSION_SPLIT_MIN, start_date=None,
                              STOCK_NUM_FIELD="Stack%", SIGMA_FIELD="sigma",
                              smin_lo=STAGE_SMIN_LO, smin_hi=STAGE_SMIN_HI, log_every_n_chunks=5):
    """-> (sorted session dates, {ticker: static sigma}).

    Sigma is READ, not computed: final.parquet already carries a per-row "sigma" column, the same static
    per-ticker dispersion figure every strategy reads ("сігму бери із фінал паркет"). The LATEST non-null
    value seen per ticker wins (row groups are read in file order, which is time order).

    `want` is a REQUIRED column list and raises if final.parquet is missing one. That is not defensiveness,
    it is the lesson from OPG-fade: a defensive `if col in df.columns else NaN` fallback there produced a
    notebook that ran clean, staged a column of 100% NaN, and would never have signalled. The guard WAS the
    failure."""
    import gc, time, shutil
    import pyarrow as pa
    import pyarrow.parquet as pq
    from pathlib import Path

    stage = Path(stage_dir)
    if stage.exists():
        shutil.rmtree(stage)
    stage.mkdir(parents=True, exist_ok=True)
    start_i = int(start_date.replace("-", "")) if start_date else -1

    schema = pa.schema([("ticker", pa.string()), ("smin", pa.int16()), ("stack", pa.float32())])
    writers, counts = {}, {}
    sigma_by_ticker: dict = {}

    def _writer(sdate):
        if sdate not in writers:
            writers[sdate] = pq.ParquetWriter(str(stage / f"{sdate}.parquet"), schema, compression="zstd")
            counts[sdate] = 0
        return writers[sdate]

    pf = pq.ParquetFile(input_path)
    want = ["ticker", "dt", STOCK_NUM_FIELD]
    cols = [c for c in want if c in pf.schema.names]
    for need in want:
        if need not in cols:
            raise KeyError(f"final.parquet is missing required column: {need}")
    has_sigma = SIGMA_FIELD in pf.schema.names
    if has_sigma:
        cols = cols + [SIGMA_FIELD]
    else:
        print(f"WARN: final.parquet has no '{SIGMA_FIELD}' column - sigma will be null for every ticker")
    t0 = time.time()
    total_in = total_out = 0
    print(f"START pullback stage  file={input_path}  session_split={session_split_min}min", flush=True)
    try:
        for ci in range(pf.num_row_groups):
            df = pf.read_row_group(ci, columns=cols).to_pandas()
            total_in += len(df)
            dt = pd.to_datetime(df["dt"], errors="coerce", utc=True)
            ok = dt.notna().to_numpy(copy=False)
            if not ok.any():
                continue
            dt, df = dt[ok], df.loc[ok]
            t_arr = dt.dt.hour.to_numpy(dtype="int32", copy=False) * 60 + dt.dt.minute.to_numpy(dtype="int32", copy=False)
            late = t_arr >= session_split_min
            smin = np.where(late, t_arr - 1440, t_arr).astype("int16")
            sess = dt + pd.to_timedelta(np.where(late, 1, 0), unit="D")
            sdate = (sess.dt.year.to_numpy(dtype="int32", copy=False) * 10000 + sess.dt.month.to_numpy(dtype="int32", copy=False) * 100
                     + sess.dt.day.to_numpy(dtype="int32", copy=False)).astype("int32")
            stack = pd.to_numeric(df[STOCK_NUM_FIELD], errors="coerce").to_numpy(dtype="float32", copy=False)
            keep = (smin >= smin_lo) & (smin <= smin_hi) & np.isfinite(stack)
            if start_i > 0:
                keep &= sdate >= start_i
            if has_sigma:
                sigma_col = pd.to_numeric(df[SIGMA_FIELD], errors="coerce").to_numpy(dtype="float64", copy=False)
                tick_col = df["ticker"].to_numpy(copy=False)
                sg_ok = np.isfinite(sigma_col)
                if sg_ok.any():
                    for tk, sv in zip(tick_col[sg_ok], sigma_col[sg_ok]):
                        sigma_by_ticker[str(tk)] = float(sv)
            if not keep.any():
                continue
            out = pd.DataFrame({"ticker": df["ticker"].to_numpy(copy=False)[keep].astype(str), "sdate": sdate[keep],
                                "smin": smin[keep], "stack": stack[keep]})
            for sd, part in out.groupby("sdate", sort=False):
                _writer(int(sd)).write_table(pa.Table.from_pandas(part[["ticker", "smin", "stack"]], schema=schema, preserve_index=False))
                counts[int(sd)] += len(part)
                total_out += len(part)
            del df, out
            if (ci + 1) % log_every_n_chunks == 0:
                print(f"[rg {ci + 1:>4}/{pf.num_row_groups}] in={total_in:,} staged={total_out:,} sessions={len(writers)} elapsed={time.time() - t0:.1f}s", flush=True)
                gc.collect()
    finally:
        for w in writers.values():
            w.close()
    print(f"DONE stage in={total_in:,} staged={total_out:,} sessions={len(writers)} "
          f"sigma_tickers={len(sigma_by_ticker):,} elapsed={time.time() - t0:.1f}s", flush=True)
    return sorted(counts), sigma_by_ticker


def pullback_sessions(stage_dir, *, smin_lo=STAGE_SMIN_LO, smin_hi=STAGE_SMIN_HI, min_tickers=30):
    """yields (sdate, panel) in date order, one pivoted session at a time."""
    from pathlib import Path
    for f in sorted(Path(stage_dir).glob("*.parquet"), key=lambda p: int(p.stem)):
        df = pd.read_parquet(f)
        if df["ticker"].nunique() < min_tickers:
            continue
        df = df.drop_duplicates(["ticker", "smin"], keep="last")
        tick, ti = np.unique(df["ticker"].to_numpy(), return_inverse=True)
        T = smin_hi - smin_lo + 1
        stack = np.full((tick.size, T), np.nan, np.float32)
        stack[ti, df["smin"].to_numpy().astype(np.int64) - smin_lo] = df["stack"].to_numpy()
        yield int(f.stem), panel_from_stack(tick, smin_lo, stack)


In [6]:
# -- self-test: the core and the three levels, on hand-built panels -------------------------------------
# Runs WITHOUT final.parquet. Every case plants tickers whose minute-by-minute Stack% is written out by
# hand, so the expected side, admission kind, pnl, alpha, gamma and lambda can be asserted exactly. Run
# this cell after any edit to the core or ratings cells: the two admission triggers, the dropped-window
# exclusion and the three level definitions are the parts most likely to look right and be wrong.
_LO = 235
_HI = 9 * 60 + 40
_T = _HI - _LO + 1
_CK = build_checkpoints()
_CFG = default_cfg()
_M = {"base": BASE_SMIN, "t0430": 4 * 60 + 30, "t0500": 5 * 60, "t0530": 5 * 60 + 30, "exit": 9 * 60 + 30}


def _panel(spec):
    tickers = sorted(spec)
    mat = np.full((len(tickers), _T), np.nan, np.float32)
    for i, tk in enumerate(tickers):
        for smin, v in spec[tk].items():
            mat[i, smin - _LO] = v
    return panel_from_stack(np.array(tickers), _LO, mat)


def _run(spec):
    return scan_day(_panel(spec), _CFG, _CK)


_fails = []


def _check(name, cond, detail=""):
    print(("  ok   " if cond else "  FAIL ") + name + (("  " + str(detail)) if not cond else ""))
    if not cond:
        _fails.append(name)


print("-- the trade --")
tr, sg, ex, sm = _run({"HIGH": {_M["base"]: 2.0, _M["t0430"]: 2.4, _M["t0500"]: 2.6, _M["t0530"]: 2.5, _M["exit"]: 0.5}})
_check("a stack standing too high is shorted at all three classes", len(tr) == 3 and (tr["sign"] == "pos").all(), tr)
_r = tr.set_index("cls").loc["T0430"]
_check("T0430 pnl = -(exit - entry) = +1.9", abs(float(_r.pnl) - 1.9) < 1e-5, _r.pnl)
_check("admitted by level (the 0.4 move alone would not)", _r.kind == "level", _r.kind)

tr, _, _, _ = _run({"LOW": {_M["base"]: -2.0, _M["t0430"]: -2.5, _M["exit"]: -0.2}})
_check("a stack standing too low is bought, pnl = +2.3",
       tr.iloc[0]["sign"] == "neg" and abs(float(tr.iloc[0].pnl) - 2.3) < 1e-5, tr)

tr, _, _, _ = _run({"MOVE": {_M["base"]: -1.8, _M["t0430"]: 0.4, _M["exit"]: 0.0}})
_check("a fresh 2.2pp move admits a stack whose 0.4 level would not", len(tr) == 1 and tr.iloc[0]["kind"] == "move", tr)

tr, _, _, _ = _run({"RIDE": {_M["base"]: -3.0, _M["t0430"]: -0.5, _M["exit"]: 0.0}})
_check("a 2.5pp move AGAINST the displacement is rejected (a ride, not a pullback)", len(tr) == 0, tr)

tr, _, _, _ = _run({"RIDE2": {_M["base"]: -5.0, _M["t0430"]: -2.5, _M["exit"]: 0.0}})
_check("...but it still enters as 'level' when the displacement clears on its own",
       len(tr) == 1 and tr.iloc[0]["kind"] == "level" and tr.iloc[0]["sign"] == "neg", tr)

tr, _, _, _ = _run({"BOTH": {_M["base"]: 0.5, _M["t0430"]: 3.0, _M["exit"]: 0.0}})
_check("both triggers -> kind = both", tr.iloc[0]["kind"] == "both", tr)

tr, _, _, _ = _run({"FLAT": {_M["base"]: 0.1, _M["t0430"]: 1.5, _M["exit"]: 0.0}})
_check("1.5pp is under the 2.0 trigger -> nothing", len(tr) == 0, tr)

tr, _, _, _ = _run({"DROP": {DROPPED_LO_SMIN: 99.0, DROPPED_HI_SMIN: 99.0, _M["base"] + 2: 1.0,
                             _M["t0430"]: 2.5, _M["exit"]: 0.0}})
_check("the 03:58-04:02 dropped window never becomes the 04:04 reference",
       len(tr) == 1 and abs(float(tr.iloc[0].base_dev) - 1.0) < 1e-5 and int(tr.iloc[0].base_minute) == _M["base"] + 2, tr)

tr, _, _, _ = _run({"NOBASE": {_M["t0430"]: 3.0, _M["exit"]: 0.0}})
_check("no 04:04 reference -> no trade", len(tr) == 0, tr)

tr, sg, _, _ = _run({"NOEXIT": {_M["base"]: 1.0, _M["t0430"]: 2.5}})
_check("no 09:30 exit -> no trade, but the signal still feeds alpha", len(tr) == 0 and len(sg) == 1, (tr, sg))

tr, _, _, _ = _run({"TOL": {_M["base"]: 1.0, _M["t0500"] + 4: 2.5, _M["exit"] - 3: 0.0}})
_check("+/-5 finds a print near but not on the checkpoint minute",
       len(tr) == 1 and int(tr.iloc[0].entry_minute) == _M["t0500"] + 4 and int(tr.iloc[0].exit_minute) == _M["exit"] - 3, tr)

tr, _, _, _ = _run({"FAR": {_M["base"]: 1.0, _M["t0500"] + 6: 2.5, _M["exit"]: 0.0}})
_check("6 minutes away is outside the tolerance", len(tr) == 0, tr)

tr, _, _, _ = _run({"LOSS": {_M["base"]: 1.0, _M["t0430"]: 2.5, _M["exit"]: 4.0}})
_check("a stack that ran further away is a loss, pnl = -1.5", abs(float(tr.iloc[0].pnl) + 1.5) < 1e-5, tr)

tr, _, _, _ = _run({"BROKEN": {_M["base"]: 1.0, _M["t0430"]: 500.0, _M["exit"]: 0.0}})
_check("a print beyond max_abs_pp is not a price", len(tr) == 0, tr)

print("\n-- ALPHA: RMS of the reading at each class threshold, per sign, PER CLASS --")
# Three sessions. At T0430 the ticker reads +3, +4, -5 -> alpha_pos = rms(3,4) = 3.5355, alpha_neg = 5.
_days = []
for i, (a, b) in enumerate([(3.0, 1.0), (4.0, 1.0), (-5.0, 1.0)]):
    _days.append((20260101 + i, _panel({"A": {_M["base"]: 0.0, _M["t0430"]: a, _M["t0500"]: b,
                                              _M["exit"]: 0.0}})))
_tr, _sg, _ex, _sm = scan_days(iter(_days), _CFG, _CK, progress_every=0)
_al = compute_alpha(_sg, default_cfg(alpha_min_n=1))
_a0430 = _al[(_al.ticker == "A") & (_al.cls == "T0430")].iloc[0]
_check("alpha_pos(T0430) = rms(3,4) = 3.5355", abs(float(_a0430.alpha_pos) - math.sqrt((9 + 16) / 2)) < 1e-4, _a0430.alpha_pos)
_check("alpha_neg(T0430) = rms(5) = 5", abs(float(_a0430.alpha_neg) - 5.0) < 1e-4, _a0430.alpha_neg)
_a0500 = _al[(_al.ticker == "A") & (_al.cls == "T0500")].iloc[0]
_check("alpha is PER CLASS: T0500 reads 1.0, not 3.5355", abs(float(_a0500.alpha_pos) - 1.0) < 1e-4, _a0500.alpha_pos)
_check("alpha is gate-free (the 1.0 readings never traded but still count)", int(_a0500.alpha_n_pos) == 3, _a0500.alpha_n_pos)

print("\n-- GAMMA: the max the stack reached between the class and 09:30, per sign --")
# One session: between 04:30 and 09:30 the stack touches +6 and -2; between 05:30 and 09:30 only +1/-2.
_spec = {"G": {_M["base"]: 0.0, _M["t0430"]: 1.0, 5 * 60 + 10: 6.0, _M["t0530"]: 1.0,
               7 * 60: -2.0, _M["exit"]: 0.0}}
_days = [(20260101 + i, _panel(_spec)) for i in range(3)]
_tr, _sg, _ex, _sm = scan_days(iter(_days), _CFG, _CK, progress_every=0)
_gm = compute_gamma(_ex, _CFG)
_g430 = _gm[(_gm.ticker == "G") & (_gm.cls == "T0430")].iloc[0]
_check("gamma_pos(T0430) = 6 (the 05:10 spike is inside 04:30..09:30)", abs(float(_g430.gamma_pos) - 6.0) < 1e-4, _g430.gamma_pos)
_check("gamma_neg(T0430) = 2 (the 07:00 dip, as a magnitude)", abs(float(_g430.gamma_neg) - 2.0) < 1e-4, _g430.gamma_neg)
_g530 = _gm[(_gm.ticker == "G") & (_gm.cls == "T0530")].iloc[0]
_check("gamma_pos(T0530) = 1 - the 05:10 spike is BEFORE this class starts",
       abs(float(_g530.gamma_pos) - 1.0) < 1e-4, _g530.gamma_pos)
_check("gamma_n counts sessions", int(_g430.gamma_n) == 3, _g430.gamma_n)

print("\n-- LAMBDA: RMS of the per-session max |Stack%| over 04:04..09:30, only sessions >= min_dev --")
# Three sessions with session maxima 3.0, 4.0 and 1.5. The 1.5 one is below the 2.0 floor and drops
# out, so lambda = rms(3,4) = 3.5355 over 2 sessions. (Under the old 0.7 floor it would have counted,
# which is exactly what this case now pins.)
_days = []
for i, peak in enumerate([3.0, 4.0, 1.5]):
    _days.append((20260101 + i, _panel({"L": {_M["base"]: 0.0, 6 * 60: peak, _M["exit"]: 0.0}})))
_tr, _sg, _ex, _sm = scan_days(iter(_days), _CFG, _CK, progress_every=0)
_lm = compute_lambda(_sm, _CFG)
_l = _lm[_lm.ticker == "L"].iloc[0]
_check("lambda = rms(3,4) = 3.5355, the 1.5 session excluded by the 2.0 floor",
       abs(float(_l["lambda"]) - math.sqrt((9 + 16) / 2)) < 1e-4, _l["lambda"])
_check("lambda_n = 2 qualifying sessions", int(_l.lambda_n) == 2, _l.lambda_n)
_check("lambda is one value for all classes (no cls column)", "cls" not in _lm.columns, _lm.columns.tolist())

print("\n-- the anomaly trim --")
_vals = [1.0] * 20 + [500.0]
_check("one broken reading does not define the RMS", _rms(_vals, 0.99, 1) < 1.2, _rms(_vals, 0.99, 1))
_check("with too few readings nothing is trimmed", abs(_rms([3.0, 4.0], 0.99, 1) - math.sqrt(12.5)) < 1e-6)

print("")
assert not _fails, "self-test failures: %s" % _fails
print("PullBack self-test: all checks passed")


-- the trade --
  ok   a stack standing too high is shorted at all three classes
  ok   T0430 pnl = -(exit - entry) = +1.9
  ok   admitted by level (the 0.4 move alone would not)
  ok   a stack standing too low is bought, pnl = +2.3
  ok   a fresh 2.2pp move admits a stack whose 0.4 level would not
  ok   a 2.5pp move AGAINST the displacement is rejected (a ride, not a pullback)
  ok   ...but it still enters as 'level' when the displacement clears on its own
  ok   both triggers -> kind = both
  ok   1.5pp is under the 2.0 trigger -> nothing
  ok   the 03:58-04:02 dropped window never becomes the 04:04 reference
  ok   no 04:04 reference -> no trade
  ok   no 09:30 exit -> no trade, but the signal still feeds alpha
  ok   +/-5 finds a print near but not on the checkpoint minute
  ok   6 minutes away is outside the tolerance
  ok   a stack that ran further away is a loss, pnl = -1.5
  ok   a print beyond max_abs_pp is not a price

-- ALPHA: RMS of the reading at each class threshold, pe

In [7]:
from pathlib import Path
import os, gzip, json, datetime


def _resolve_orion_paths(strategy_code: str):
    final_env = os.environ.get("FINAL_PARQUET_PATH")
    sig_env = os.environ.get("SIGNALS_DIR")
    orion_env = os.environ.get("ORION_HOME")

    final_path = Path(final_env).expanduser().resolve() if final_env else None
    signals_base = Path(sig_env).expanduser().resolve() if sig_env else None

    if (final_path is None or signals_base is None) and orion_env:
        orion_home = Path(orion_env).expanduser().resolve()
        if final_path is None:
            final_path = (orion_home / "CRACEN" / "final.parquet").resolve()
        if signals_base is None:
            signals_base = (orion_home / "signals").resolve()

    if final_path is None or signals_base is None:
        here = Path.cwd().resolve()
        orion_home = None
        for parent in [here] + list(here.parents):
            if parent.name.lower() == "orion":
                orion_home = parent
                break
            cand = parent / "OriON"
            if cand.exists() and cand.is_dir():
                orion_home = cand.resolve()
                break
        if orion_home is None:
            raise RuntimeError("Cannot locate OriON. Set ORION_HOME env var (recommended).")
        if final_path is None:
            final_path = (orion_home / "CRACEN" / "final.parquet").resolve()
        if signals_base is None:
            signals_base = (orion_home / "signals").resolve()

    out_dir = (signals_base / strategy_code.lower()).resolve()
    out_dir.mkdir(parents=True, exist_ok=True)
    if not final_path.exists():
        raise FileNotFoundError(f"FINAL parquet not found: {final_path}")
    return final_path, out_dir


# -- runner ---------------------------------------------------------------------------------------------------
# Strategy code "pullback" -> signals/pullback/pullback_rolling_perf.json.gz.
#
# NOTE: NO BRIDGE SERVICE READS THIS FILE YET. There is no PullBackGate, PullBackRatingsService or
# PullBackRollingPerfParser in TradingBridgeApi. The row/meta shape stays close to the sibling families'
# so that job is a short port - but a porter must read the alpha/gamma/lambda definitions in the ratings
# cell first, because all three MEAN something different here and a parser that assumes the family
# semantics would publish the right numbers under the wrong names.
FINAL_PATH, OUT_DIR = _resolve_orion_paths("pullback")

STAGE_DIR = OUT_DIR.parent.parent / "_work" / "pullback_stage_v2"
STAGE_DIR.parent.mkdir(parents=True, exist_ok=True)
SIGMA_SIDECAR = STAGE_DIR.parent / "pullback_stage_sigma.json"

_stage_ok = STAGE_DIR.exists() and any(STAGE_DIR.glob("*.parquet")) and SIGMA_SIDECAR.exists() and \
    min(p.stat().st_mtime for p in STAGE_DIR.glob("*.parquet")) >= FINAL_PATH.stat().st_mtime
if _stage_ok:
    print("stage is at least as new as final.parquet (incl. its sigma sidecar) - reusing", STAGE_DIR)
    sigma_by_ticker = json.loads(SIGMA_SIDECAR.read_text(encoding="utf-8"))
else:
    _, sigma_by_ticker = pullback_stage_by_session(str(FINAL_PATH), str(STAGE_DIR))
    SIGMA_SIDECAR.write_text(json.dumps(sigma_by_ticker), encoding="utf-8")
    if not sigma_by_ticker:
        print("WARN: fresh stage still produced no sigma for any ticker - check final.parquet's own "
              "'sigma' column")

CFG = default_cfg()
CHECKPOINTS = build_checkpoints()

trades, signals, excursions, session_max = scan_days(pullback_sessions(str(STAGE_DIR)), CFG, CHECKPOINTS)
print(f"trades: {len(trades):,} over {trades['sdate'].nunique() if len(trades) else 0} sessions, "
      f"{trades['ticker'].nunique() if len(trades) else 0} tickers")

summary, alpha, gamma, lam = build_summary(trades, signals, excursions, session_max, CFG, sigma_by_ticker)
print(f"alpha rows (ticker x class): {len(alpha):,} | gamma rows: {len(gamma):,} | lambda tickers: {len(lam):,}")
summary.to_csv(OUT_DIR / "pullback_ratings.csv", index=False)
alpha.to_csv(OUT_DIR / "pullback_alpha.csv", index=False)
gamma.to_csv(OUT_DIR / "pullback_gamma.csv", index=False)
lam.to_csv(OUT_DIR / "pullback_lambda.csv", index=False)

trade_dates_sorted = sorted(trades["trade_date"].unique().tolist()) if len(trades) else []
recent_dates = [str(s) for s in trade_dates_sorted][-255:]
date_index = {int(d): i for i, d in enumerate(recent_dates)}
WINDOWS_DAYS = [5, 20, 40, 65]
CLASS_ORDER = list(ENTRY_CLASSES)


def _r4(v):
    return None if v is None or (isinstance(v, float) and pd.isna(v)) or pd.isna(v) else round(float(v), 4)


_alpha_idx = {(r.ticker, r.cls): r for r in alpha.itertuples(index=False)} if len(alpha) else {}
_gamma_idx = {(r.ticker, r.cls): r for r in gamma.itertuples(index=False)} if len(gamma) else {}
# NOT itertuples: "lambda" is a Python keyword, so pandas silently renames that attribute to a
# positional `_2` and `getattr(row, "lambda", None)` returns None for every ticker - which is exactly
# what shipped on the first real run (lambda_n populated, lambda null, 7,216 rows). Same shape as the
# `lam` name collision in PairFlux. A plain dict cannot be renamed.
_lam_idx = {r["ticker"]: (r["lambda"], int(r["lambda_n"])) for _, r in lam.iterrows()} if len(lam) else {}

# Every (ticker, class) that has ANY level, whether or not it ever traded - alpha and gamma are
# statements about the stack, not about the trade history, so withholding them until a trade existed
# would hide exactly the tickers a gate most needs to judge.
_keys = set(_alpha_idx) | set(_gamma_idx)
by_ticker_levels: dict = {}
for tk, cls in sorted(_keys):
    a = _alpha_idx.get((tk, cls))
    g = _gamma_idx.get((tk, cls))
    by_ticker_levels.setdefault(tk, []).append({
        "cls": cls,
        "alpha_pos": _r4(getattr(a, "alpha_pos", None)) if a is not None else None,
        "alpha_neg": _r4(getattr(a, "alpha_neg", None)) if a is not None else None,
        "alpha_n_pos": int(a.alpha_n_pos) if a is not None else 0,
        "alpha_n_neg": int(a.alpha_n_neg) if a is not None else 0,
        "gamma_pos": _r4(getattr(g, "gamma_pos", None)) if g is not None else None,
        "gamma_neg": _r4(getattr(g, "gamma_neg", None)) if g is not None else None,
        "gamma_pos_p90": _r4(getattr(g, "gamma_pos_p90", None)) if g is not None else None,
        "gamma_neg_p90": _r4(getattr(g, "gamma_neg_p90", None)) if g is not None else None,
        "gamma_n": int(g.gamma_n) if g is not None else 0,
    })

by_ticker_ratings: dict = {}
for _, r in summary.iterrows():
    by_ticker_ratings.setdefault(r["ticker"], []).append({
        "cls": r["cls"], "sign": r["sign"], "total": int(r["total"]), "wins": int(r["wins"]),
        "win_rate": _r4(r["win_rate"]), "win_rate_lb": _r4(r["win_rate_lb"]),
        "mean_pnl": _r4(r["mean_pnl"]), "median_pnl": _r4(r["median_pnl"]), "pnl_lb": _r4(r["pnl_lb"]),
        # sign-matched copies of the per-class levels, so a consumer reading one rating row needs no join
        "alpha": _r4(r["alpha"]), "gamma": _r4(r["gamma"]), "gamma_p90": _r4(r["gamma_p90"]),
    })

ROLLING_PATH = OUT_DIR / "pullback_rolling_perf.json.gz"
trades_by_ticker = dict(list(trades.groupby("ticker"))) if len(trades) else {}
rows = []
for tk in sorted(set(by_ticker_levels) | set(by_ticker_ratings) | set(trades_by_ticker)):
    lm = _lam_idx.get(tk)
    g = trades_by_ticker.get(tk)
    rows.append({
        "ticker": tk,
        "sigma": _r4(sigma_by_ticker.get(tk)),
        "lambda": _r4(lm[0]) if lm is not None else None,
        "lambda_n": lm[1] if lm is not None else 0,
        "levels": by_ticker_levels.get(tk, []),
        "ratings": by_ticker_ratings.get(tk, []),
        "trades": [] if g is None else [{
            "cls": r["cls"], "sign": r["sign"], "date": date_index.get(int(r["trade_date"])),
            "entry_dev": _r4(r["entry_dev"]), "pnl_pct": _r4(r["pnl"]),
            "status": "win" if r["pnl"] > 0 else "loss", "kind": r["kind"],
            "base_dev": _r4(r["base_dev"]), "move_dev": _r4(r["move_dev"]),
            "entry_minute_idx": int(r["entry_minute"]), "exit_minute_idx": int(r["exit_minute"]),
        } for _, r in g.iterrows() if int(r["trade_date"]) in date_index],
    })

with gzip.open(ROLLING_PATH, "wt", encoding="utf-8", newline="\n") as f:
    json.dump({
        "meta": {
            "version": "pullback-rolling-perf-v2",
            "generated_at": datetime.datetime.utcnow().isoformat() + "Z",
            "most_recent_session": recent_dates[-1] if recent_dates else None,
            "sessions_seen": len(recent_dates),
            "windows_days": WINDOWS_DAYS,
            "recent_dates": recent_dates,
            "position_usd": 1000,
            "min_dev": CFG["min_dev"],
            "classes": CLASS_ORDER,
            "pnl_basis": "raw Stack% move. A stack standing more than min_dev away from yesterday's close "
                         "at the reversal checkpoint (04:30 / 05:00 / 05:30, +/-5min) is FADED - Short what "
                         "stands too high, Long what stands too low - and held to the 09:30 open. "
                         "pnl = -sign(displacement) * (exit - entry). Admission is either the displacement "
                         "itself (an event that began before 04:04) or a same-direction move of min_dev "
                         "since the 04:04 reference; `kind` on each trade says which.",
            "alpha_basis": "PER CLASS, per sign: RMS of the displacement at that class's own checkpoint "
                           "over every session the ticker printed there, gate-free, with the top "
                           "quantile of |readings| trimmed. NOT the sibling families' modal per-ticker "
                           "alpha.",
            "gamma_basis": "PER CLASS, per sign: the MAXIMUM Stack% reached between that class's "
                           "checkpoint and 09:30 (gamma_neg is the magnitude of the lowest). A raw max "
                           "is defined by one session - gamma_*_p90 and gamma_n ship beside it. NOT the "
                           "sibling families' lowest-profitable-level search.",
            "lambda_basis": "per ticker, one value for all classes: RMS of the per-session maximum "
                            "|Stack%| over 04:04..09:30, counting only sessions whose maximum reached "
                            "min_dev. NOT the sibling families' last-print-before-the-close statistic.",
            "sigma_basis": "read from final.parquet's own 'sigma' column, unchanged.",
            "checkpoints": {k: [v.smin, v.tol] for k, v in CHECKPOINTS.items()},
            "base_minute": [BASE_SMIN, BASE_TOL],
            "dropped_window": [DROPPED_LO_SMIN, DROPPED_HI_SMIN],
        },
        "rows": rows,
    }, f, ensure_ascii=False)

print("written ->", OUT_DIR)
for _f in ("pullback_ratings.csv", "pullback_alpha.csv", "pullback_gamma.csv", "pullback_lambda.csv",
           ROLLING_PATH.name):
    print("   ", _f)


START pullback stage  file=C:\datum-api-examples-main\OriON\CRACEN\final.parquet  session_split=900min


[rg    5/59] in=10,000,000 staged=897,001 sessions=64 elapsed=10.5s


[rg   10/59] in=18,540,244 staged=1,679,288 sessions=64 elapsed=21.5s


[rg   15/59] in=28,540,244 staged=2,487,483 sessions=64 elapsed=33.8s


[rg   20/59] in=38,483,751 staged=3,185,366 sessions=64 elapsed=45.5s


...

  scanned 50 sessions (last 20260928) trades so far 34,443


  scanned 60 sessions (last 20261012) trades so far 39,812


trades: 42,604 over 64 sessions, 2496 tickers


alpha rows (ticker x class): 14,530 | gamma rows: 22,794 | lambda tickers: 6,331


written -> C:\datum-api-examples-main\OriON\signals\pullback
    pullback_ratings.csv
    pullback_alpha.csv
    pullback_gamma.csv
    pullback_lambda.csv
    pullback_rolling_perf.json.gz


In [8]:
# -- THE ANALYSIS the operator asked for: which reversal start time is best? ---------------------------
# "Проведи аналіз час початку розвороту це 3 класи" - the question is not "does PullBack work" but "of
# 04:30 / 05:00 / 05:30, which is the moment to start reversing". Everything below is cut by class.
#
# TWO THINGS ARE REPORTED THAT A NAIVE READ WOULD SKIP, and both change the answer:
#
#   TRIMMED means, at the 99th percentile of |pnl|. On the sibling 04:00 study the top 1% of trades
#   carried 60% of the result; an untrimmed mean on pre-market stacks measures a handful of 200%
#   gappers, not a strategy.
#
#   SESSION-LEVEL statistics, not just per-trade ones. Events inside one morning are heavily correlated
#   - a market-wide gap down makes every short win at once - so a per-trade confidence bound over
#   thousands of rows is far tighter than the evidence actually is. "sess+" is the honest sample size.
import numpy as np
import pandas as pd


def _wilson(k, n, z=1.96):
    if n <= 0:
        return float("nan")
    p = k / n
    den = 1 + z * z / n
    c = p + z * z / (2 * n)
    r = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return (c - r) / den


def _block(g, trim_q=0.99):
    pnl = g["pnl"].to_numpy(float)
    if pnl.size == 0:
        return None
    cap = np.quantile(np.abs(pnl), trim_q) if pnl.size > 20 else np.inf
    tp = np.clip(pnl, -cap, cap)
    n = pnl.size
    wins = int((pnl > 0).sum())
    se = tp.std(ddof=1) / math.sqrt(n) if n > 1 else float("nan")
    per_session = g.groupby("sdate")["pnl"].mean()
    return {
        "n": n,
        "tickers": g["ticker"].nunique(),
        "sessions": g["sdate"].nunique(),
        "per_day": round(n / max(1, g["sdate"].nunique()), 1),
        "mean": round(float(pnl.mean()), 4),
        "mean_trim": round(float(tp.mean()), 4),
        "lb95_trim": round(float(tp.mean() - 1.96 * se), 4) if n > 1 else None,
        "median": round(float(np.median(pnl)), 4),
        "win%": round(100.0 * wins / n, 1),
        "win_lb%": round(100.0 * _wilson(wins, n), 1),
        "sess+": f"{int((per_session > 0).sum())}/{per_session.size}",
        "sess_mean": round(float(per_session.mean()), 4),
    }


def _table(df, by, title, trim_q=0.99):
    rows = []
    for key, g in df.groupby(by, sort=True):
        b = _block(g, trim_q)
        if b is None:
            continue
        if not isinstance(key, tuple):
            key = (key,)
        rows.append({**dict(zip(by if isinstance(by, list) else [by], key)), **b})
    out = pd.DataFrame(rows)
    print("\n== " + title + " " + "=" * max(0, 86 - len(title)))
    print(out.to_string(index=False) if len(out) else "  (no rows)")
    return out


if not len(trades):
    print("No trades - nothing to analyse. Check that the stage produced sessions and that final.parquet "
          "covers the 03:55-09:40 band.")
else:
    print(f"PullBack: {len(trades):,} trades, {trades['ticker'].nunique():,} tickers, "
          f"{trades['sdate'].nunique()} sessions, trigger min_dev={CFG['min_dev']}pp")

    by_class = _table(trades, ["cls"], "BY REVERSAL START (the operator's question)")
    _table(trades, ["cls", "sign"], "BY CLASS x SIDE  (pos = stood too high = SHORT)")
    _table(trades, ["cls", "kind"], "BY CLASS x ADMISSION  (level = began before 04:04, move = after)")

    # Bands start at the trigger itself - with min_dev at 2.0 the old 0.7-1 / 1-1.5 rows would be
    # empty, and the interesting structure is now ABOVE 2, which is where the first run found the
    # edge (+0.22 at 2.5-4, +0.47 at 4-7, +1.31 at 7+).
    bins = [CFG["min_dev"], 2.5, 3.0, 4.0, 5.5, 8.0, np.inf]
    labels = ["min-2.5", "2.5-3", "3-4", "4-5.5", "5.5-8", "8+"]
    band = trades.assign(dev_band=pd.cut(trades["entry_dev"].abs(), bins=bins, labels=labels,
                                         right=False, include_lowest=True))
    _table(band.dropna(subset=["dev_band"]), ["cls", "dev_band"], "BY CLASS x |DISPLACEMENT| AT ENTRY")

    sess = sorted(trades["sdate"].unique())
    if len(sess) >= 6:
        cut = len(sess) // 2
        train, test = set(sess[:cut]), set(sess[cut:])
        tr_tr = trades[trades["sdate"].isin(train)]
        tr_te = trades[trades["sdate"].isin(test)]
        best = None
        print("\n== TRAIN / TEST BY SESSION " + "=" * 60)
        for cls in ENTRY_CLASSES:
            a, b = _block(tr_tr[tr_tr["cls"] == cls]), _block(tr_te[tr_te["cls"] == cls])
            if a is None or b is None:
                continue
            print(f"  {cls}: train mean_trim {a['mean_trim']:+.4f} (n={a['n']:,})   |   "
                  f"test mean_trim {b['mean_trim']:+.4f}  lb95 {b['lb95_trim']}  win {b['win%']}%  "
                  f"sess+ {b['sess+']}  (n={b['n']:,})")
            if best is None or a["mean_trim"] > best[1]:
                best = (cls, a["mean_trim"], b)
        if best is not None:
            print(f"\n  Class chosen on TRAIN alone = {best[0]} -> out of sample "
                  f"{best[2]['mean_trim']:+.4f} pp/trade, lb95 {best[2]['lb95_trim']}, "
                  f"win {best[2]['win%']}%, sessions positive {best[2]['sess+']}")
    else:
        print("\n(too few sessions for a train/test split - need at least 6)")

    if len(by_class):
        by_class.to_csv(OUT_DIR / "pullback_class_analysis.csv", index=False)
        print("\nwritten ->", OUT_DIR / "pullback_class_analysis.csv")

    print("\n-- the three levels, pooled across tickers (a sanity read, not a result) --")
    if len(alpha):
        print("ALPHA  (RMS displacement at the class threshold)")
        print(alpha.groupby("cls")[["alpha_pos", "alpha_neg"]].median().round(3).to_string())
    if len(gamma):
        print("\nGAMMA  (max excursion from the class to 09:30)")
        print(gamma.groupby("cls")[["gamma_pos", "gamma_neg", "gamma_pos_p90", "gamma_neg_p90"]]
              .median().round(3).to_string())
    if len(lam):
        print(f"\nLAMBDA median {lam['lambda'].median():.3f} over {len(lam):,} tickers "
              f"(median {lam['lambda_n'].median():.0f} qualifying sessions each)")

    print("\nREAD THIS BEFORE BELIEVING ANY NUMBER ABOVE:")
    print("  * entry and exit are both taken at a PRINT, with no spread paid. Pre-market spreads on these")
    print("    names are wide - on the sibling 04:00 study each half-spread was worth more than the whole")
    print("    edge (taker -0.26 vs mid +0.31 on a median 1.07% spread). Treat every mean here as an")
    print("    upper bound that execution has not been subtracted from yet.")
    print("  * 'sess+' is the real sample size, not 'n'.")


PullBack: 42,604 trades, 2,496 tickers, 64 sessions, trigger min_dev=2.0pp

== BY REVERSAL START (the operator's question) ===========================================
  cls     n  tickers  sessions  per_day   mean  mean_trim  lb95_trim  median  win%  win_lb% sess+  sess_mean
T0430 14739     2171        64    230.3 0.6483     0.5898     0.5057  0.2732  53.6     52.8 49/64     0.7273
T0500 14031     2056        64    219.2 0.6080     0.5609     0.4800  0.2658  53.4     52.6 44/64     0.6495
T0530 13834     2009        64    216.2 0.4878     0.4411     0.3630  0.1845  52.7     51.9 46/64     0.5270

== BY CLASS x SIDE  (pos = stood too high = SHORT) =======================================
  cls sign    n  tickers  sessions  per_day   mean  mean_trim  lb95_trim  median  win%  win_lb% sess+  sess_mean
T0430  neg 7622     1660        64    119.1 0.2900     0.2556     0.1536  0.0000  49.7     48.6 44/64     0.3769
T0430  pos 7117     1722        64    111.2 1.0319     0.9638     0.8248  0.540

  T0530: train mean_trim +0.4767 (n=7,339)   |   test mean_trim +0.3961  lb95 0.2895  win 51.8%  sess+ 22/32  (n=6,495)

  Class chosen on TRAIN alone = T0500 -> out of sample +0.4118 pp/trade, lb95 0.2986, win 52.1%, sessions positive 21/32

written -> C:\datum-api-examples-main\OriON\signals\pullback\pullback_class_analysis.csv

-- the three levels, pooled across tickers (a sanity read, not a result) --
ALPHA  (RMS displacement at the class threshold)
       alpha_pos  alpha_neg
cls                        
T0430      2.081      2.163
T0500      2.263      2.312
T0530      2.326      2.421

GAMMA  (max excursion from the class to 09:30)
       gamma_pos  gamma_neg  gamma_pos_p90  gamma_neg_p90
cls                                                      
T0430      6.114      5.021          2.544          2.252
T0500      6.031      4.987          2.538          2.240
T0530      5.978      4.960          2.515          2.233

LAMBDA median 4.012 over 6,331 tickers (median 14 qualifying se